# 4.9 Correlation & Covariance

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.09_correlation-and-covariance.ipynb)

Companion code for Section 4.9 of the book. Run the cells in order from top to bottom.

**Example 4.9.1**

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 500
age    = np.random.normal(40, 10, n)
income = 1500 * age + np.random.normal(0, 15000, n)   # Correlated with age
random = np.random.normal(0, 1, n)                     # Independent

df = pd.DataFrame({'age': age, 'income': income, 'random': random})

# Covariance matrix
cov_matrix = df.cov()
print('Covariance matrix:')
print(cov_matrix.round(2))
print(f'\nCov(age, income) = {np.cov(age, income)[0,1]:,.2f}')
print(f'Interpretation: large positive - income increases with age')
print(f'Problem: magnitude in income² units - hard to interpret')

**Example 4.9.2**

In [ ]:
from scipy import stats
import pandas as pd
import numpy as np

# Pearson r and its p-value
r_age_income, p_age_income = stats.pearsonr(age, income)
r_age_random, p_age_random = stats.pearsonr(age, random)

print(f'Pearson r(age, income): {r_age_income:.4f}  p={p_age_income:.2e}')
print(f'Pearson r(age, random): {r_age_random:.4f}  p={p_age_random:.4f}')

# Full correlation matrix
print('\nCorrelation matrix:')
print(df.corr().round(4))

# Identify strong correlations
threshold = 0.5
corr = df.corr()
print(f'\nCorrelation pairs with |r| > {threshold}:')
for col in corr.columns:
    for row in corr.index:
        if col != row and abs(corr.loc[row, col]) > threshold:
            print(f'  {row} ↔ {col}: r={corr.loc[row,col]:.4f}')

# Note: Pearson can MISS non-linear relationships
x = np.linspace(-3, 3, 200)
y_nonlinear = x**2 + np.random.normal(0, 0.5, 200)  # Perfect quadratic - no linear signal
r_nonlinear, _ = stats.pearsonr(x, y_nonlinear)
print(f'\nX vs X² relationship: Pearson r={r_nonlinear:.4f}  (misses the pattern!)')

**Example 4.9.3**

In [ ]:
from scipy import stats
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
df = housing.frame
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# ── Spearman rank correlation - captures monotonic relationships ──
# Converts values to ranks first - robust to outliers and non-linearity
spearman_corr = X.apply(lambda col: stats.spearmanr(col, y).statistic).sort_values(key=abs, ascending=False)
print('Spearman correlation with target (house value):')
print(spearman_corr.round(4))

# ── Mutual Information - captures ANY relationship ────────────────
# MI(X; Y) = 0 means X and Y are independent
# MI(X; Y) > 0 means knowing X gives information about Y
# MI is symmetric and captures non-linear relationships
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_df = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
print('\nMutual Information with target:')
print(mi_df.round(4))

# Compare Pearson, Spearman, and MI rankings
pearson  = X.corrwith(y).abs().rename('pearson_|r|')
spearman = spearman_corr.abs().rename('spearman_|r|')
mi       = mi_df.rename('mutual_info')

comparison = pd.concat([pearson, spearman, mi], axis=1).sort_values('mutual_info', ascending=False)
print('\nFeature ranking comparison (Pearson vs Spearman vs MI):')
print(comparison.round(4))

print('\nMI captures non-linear patterns Pearson misses.')
print('Use MI as primary feature importance metric for EDA.')